## 1. Import Required Libraries

# Nestlé HR Policy Chatbot (Assignment Solution)

## Overview
This notebook implements a conversational chatbot that answers user inquiries using Nestlé's HR policy PDF. It uses **PyPDFLoader** to load the document, **Chroma DB** with **OpenAI embeddings** for vector representations, **GPT-3.5 Turbo** for the question-answering system, and **Gradio** for a user-friendly interface.

### Assignment Requirements Implemented:
- Import essential tools and set up OpenAI API environment
- Load Nestlé's HR policy using **PyPDFLoader** and split for easy processing
- Create vector representations using **Chroma DB** and **OpenAI embeddings**
- Build question-answering system using **GPT-3.5 Turbo**
- Prompt template to guide the chatbot
- **Gradio** user-friendly chatbot interface

### Requirements:
- OpenAI API key (get from https://platform.openai.com/api-keys)
- Nestlé HR policy PDF in `./Dataset/the_nestle_hr_policy_pdf_2012.pdf`

In [1]:
# Install required packages (per assignment: Python libraries, OpenAI, Gradio, Chroma, PyPDF)
import subprocess
import sys

packages = [
    'langchain',
    'langchain-community',
    'langchain-openai',
    'openai',
    'pypdf',
    'chromadb',
    'numpy',
    'python-dotenv',
    'gradio'
]

for package in packages:
    try:
        mod = package.replace('-', '_')
        if mod == 'langchain_community':
            __import__('langchain_community.document_loaders')
        else:
            __import__(mod)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

# Import all required libraries
import os
import numpy as np
from pathlib import Path
from typing import List, Dict, Any



✓ langchain already installed
✓ langchain-community already installed
✓ langchain-openai already installed
✓ openai already installed
✓ pypdf already installed
Installing chromadb...
✓ chromadb installed
✓ numpy already installed
Installing python-dotenv...
✓ python-dotenv installed
✓ gradio already installed


In [2]:
# LangChain imports: text splitter, embeddings, LLM, prompts, chains, document loaders, vector store
try:
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_core.prompts import PromptTemplate
    from langchain_classic.chains import RetrievalQA
    from langchain_community.vectorstores import Chroma
    print("✓ Using langchain_community + langchain_openai")
except ImportError:
    try:
        from langchain_community.document_loaders import PyPDFLoader
        from langchain_community.vectorstores import Chroma
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        from langchain.embeddings.openai import OpenAIEmbeddings
        from langchain.chat_models import ChatOpenAI
        from langchain.prompts import PromptTemplate
        from langchain_classic.chains import RetrievalQA
        print("✓ Using legacy LangChain imports")
    except ImportError as e:
        raise ImportError(f"Install: pip install langchain langchain-community langchain-openai chromadb. Error: {e}")

print("✓ All libraries imported successfully!")

✓ Using langchain_community + langchain_openai
✓ All libraries imported successfully!


In [3]:
# Configure OpenAI API
# You need to set your OpenAI API key here
# Option 1: Set it directly (for testing only - not recommended for production)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# Option 2: Use .env file or environment variable
try:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv())
    print("✓ .env file loaded (if present)")
except ImportError:
    print("⚠️ python-dotenv not available, using environment variables only")

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("\n⚠️  WARNING: OPENAI_API_KEY environment variable not set!")
    print("Please set your OpenAI API key to use this notebook.")
    print("\nTo set it:")
    print("  Using .env file: Create a .env file with OPENAI_API_KEY=your-key")
    print("  Or set environment variable: export OPENAI_API_KEY='your-key'")
    print("  Or in Python: import os; os.environ['OPENAI_API_KEY'] = 'your-key'")
    print("\nYou can get your API key from: https://platform.openai.com/api-keys")
    print("\nThe notebook will continue but AI features won't work without the API key.")
else:
    print("✓ OpenAI API Key configured successfully!")

# Define paths
dataset_path = Path("./Dataset/the_nestle_hr_policy_pdf_2012.pdf")
print(f"\nDataset path: {dataset_path}")
print(f"Dataset exists: {dataset_path.exists()}")

✓ .env file loaded (if present)
✓ OpenAI API Key configured successfully!

Dataset path: Dataset\the_nestle_hr_policy_pdf_2012.pdf
Dataset exists: True


## 2. Load and Parse HR Policy PDF

In [4]:
# Load Nestlé's HR policy using PyPDFLoader (per assignment requirement)
if dataset_path.exists():
    loader = PyPDFLoader(str(dataset_path))
    documents = loader.load()
    print(f"Loading PDF from: {dataset_path}")
    print(f"✓ PyPDFLoader: loaded {len(documents)} pages")
    print(f"Total characters: {sum(len(d.page_content) for d in documents)}")
    print(f"\nFirst 500 characters:\n{documents[0].page_content[:500]}\n...")
else:
    documents = []
    print(f"❌ PDF file not found at {dataset_path}")

Loading PDF from: Dataset\the_nestle_hr_policy_pdf_2012.pdf
✓ PyPDFLoader: loaded 8 pages
Total characters: 14322

First 500 characters:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
...


In [5]:
# Split documents for easy processing (per assignment: split for easy processing)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
document_chunks = text_splitter.split_documents(documents) if documents else []
print(f"Text split into {len(document_chunks)} chunks")
if document_chunks:
    print(f"Average chunk size: {np.mean([len(c.page_content) for c in document_chunks]):.0f} characters")

# Show sample chunks
print(f"\nSample chunks (first 3):")
print("=" * 80)
for i, doc in enumerate(document_chunks[:3]):
    content = doc.page_content
    print(f"\nChunk {i+1}:")
    print(content[:300] + "..." if len(content) > 300 else content)
    print("=" * 80)

Text split into 20 chunks
Average chunk size: 837 characters

Sample chunks (first 3):

Chunk 1:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy

Chunk 2:
Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/ne...

Chunk 3:
The Nestlé Human Resources Policy
1
At Nestlé, we recognize that our employees 
are the key to our success and nothing can be 
achieved without their engagement. 
This document encompasses the guidelines 
which constitute a solid basis for effective Human 
Resources Management throughout the Nestlé ...


## 3. Create Vector Embeddings and Store in Vector Database

In [6]:
# Create vector representations using Chroma DB and OpenAI embeddings (per assignment)
if openai_api_key and document_chunks:
    try:
        print("Creating OpenAI embeddings and Chroma DB vector store...")
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        
        # Chroma DB: store vector representations of text chunks
        persist_dir = "./chroma_hr_policy"
        vector_store = Chroma.from_documents(
            documents=document_chunks,
            embedding=embeddings,
            persist_directory=persist_dir,
            collection_name="nestle_hr_policy"
        )

        # Test retrieval
        print("\nTesting retrieval with a sample query...")
        test_query = "What is the leave policy?"
        test_results = vector_store.similarity_search(test_query, k=2)
        print(f"Found {len(test_results)} relevant documents for: '{test_query}'")
        print(f"Top result preview: {test_results[0].page_content[:200]}...")
        
    except Exception as e:
        print(f"❌ Error creating embeddings/Chroma: {str(e)}")
        vector_store = None
else:
    if not openai_api_key:
        print("❌ Cannot create embeddings without OpenAI API key")
    if not document_chunks:
        print("❌ No document chunks to embed. Load PDF first.")
    vector_store = None

Creating OpenAI embeddings and Chroma DB vector store...

Testing retrieval with a sample query...
Found 2 relevant documents for: 'What is the leave policy?'
Top result preview: Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy...


## 4. Build RAG Pipeline with LLM

In [7]:
# Prompt template to guide the chatbot (per assignment)
hr_assistant_prompt = """Use the following pieces of context about Nestlé's HR policies to answer the user's question.
If you don't find relevant information in the provided context, say "I don't have information about this in the HR policy document."

IMPORTANT: 
- Always base your answers on the provided HR policy context
- Be clear and concise in your responses
- If a policy requires further action or consultation, mention that

Context from HR Policy:
{context}

User Question: {question}

HR Assistant Response:"""

PROMPT = PromptTemplate(
    template=hr_assistant_prompt,
    input_variables=["context", "question"]
)

# Build question-answering system using GPT-3.5 Turbo (per assignment)
if vector_store and openai_api_key:
    try:
        print("Creating question-answering system with GPT-3.5 Turbo...")
        
        llm = ChatOpenAI(
            openai_api_key=openai_api_key,
            model_name="gpt-3.5-turbo",
            temperature=0.3
        )
        
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )
        
        print("✓ Question-answering system created successfully!")
        print("\nPipeline Components:")
        print("  - LLM: GPT-3.5 Turbo")
        print("  - Vector Store: Chroma DB")
        print("  - Embeddings: OpenAI")
        print("  - Retrieval: Top 3 documents")
        
    except Exception as e:
        print(f"❌ Error creating QA pipeline: {str(e)}")
        qa_chain = None
else:
    print("❌ Cannot create QA pipeline without vector store and API key")
    qa_chain = None

Creating question-answering system with GPT-3.5 Turbo...
✓ Question-answering system created successfully!

Pipeline Components:
  - LLM: GPT-3.5 Turbo
  - Vector Store: Chroma DB
  - Embeddings: OpenAI
  - Retrieval: Top 3 documents


## 5. Create Chatbot Interface

In [8]:
class HRAssistant:
   
    
    def __init__(self, qa_chain, vector_store):
        
        self.qa_chain = qa_chain
        self.vector_store = vector_store
        self.conversation_history = []
    
    def answer_question(self, question: str) -> Dict[str, Any]:
       
        if not self.qa_chain:
            return {
                "answer": "Error: RAG pipeline not initialized. Please check your OpenAI API key.",
                "sources": [],
                "error": True
            }
        
        try:
            # Get response from QA chain
            response = self.qa_chain({"query": question})
            
            # Extract source documents
            source_docs = response.get("source_documents", [])
            sources = [
                {
                    "content": doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content,
                    "full_content": doc.page_content
                }
                for doc in source_docs
            ]
            
            # Store in conversation history
            self.conversation_history.append({
                "question": question,
                "answer": response["result"],
                "sources": sources
            })
            
            return {
                "answer": response["result"],
                "sources": sources,
                "error": False
            }
        
        except Exception as e:
            return {
                "answer": f"Error processing question: {str(e)}",
                "sources": [],
                "error": True
            }
    
    def get_conversation_history(self) -> List[Dict]:
        """Get the conversation history."""
        return self.conversation_history
    
    def clear_history(self):
        """Clear the conversation history."""
        self.conversation_history = []


# Create HR Assistant instance
if qa_chain and vector_store:
    hr_assistant = HRAssistant(qa_chain, vector_store)
    print("✓ HR Assistant initialized successfully!")
    print("\nYou can now ask questions about Nestlé's HR policies.")
    print("Use: hr_assistant.answer_question('Your question here')")
else:
    hr_assistant = None
    print("❌ Could not initialize HR Assistant. Please check the setup.")

✓ HR Assistant initialized successfully!

You can now ask questions about Nestlé's HR policies.
Use: hr_assistant.answer_question('Your question here')


## 6. Test with Sample HR Queries

In [9]:
# Test the HR Assistant with sample queries
if hr_assistant:
    # Sample HR Policy Questions
    sample_queries = [
        "What is the annual leave policy?",
        "What are the employee benefits?",
        "How is sick leave handled?",
        "What is the maternity policy?",
        "What are the working hours?",
        "How are performance appraisals conducted?"
    ]
    
    print("=" * 80)
    print("HR ASSISTANT - TESTING WITH SAMPLE QUERIES")
    print("=" * 80)
    
    # Test with first 3 queries
    for i, query in enumerate(sample_queries[:3], 1):
        print(f"\n\n{'='*80}")
        print(f"QUERY {i}: {query}")
        print(f"{'='*80}\n")
        
        response = hr_assistant.answer_question(query)
        
        if not response["error"]:
            print(f"ANSWER:\n{response['answer']}\n")
            
            if response["sources"]:
                print(f"\nSOURCE DOCUMENTS ({len(response['sources'])} found):")
                print("-" * 80)
                for j, source in enumerate(response["sources"], 1):
                    print(f"\nSource {j}:")
                    print(source["content"])
        else:
            print(f"ERROR: {response['answer']}")
    
    print(f"\n\n{'='*80}")
    print("CONVERSATION HISTORY SUMMARY")
    print(f"{'='*80}")
    print(f"Total queries processed: {len(hr_assistant.get_conversation_history())}")
    
else:
    print("❌ HR Assistant not available. Please ensure all setup steps completed successfully.")

HR ASSISTANT - TESTING WITH SAMPLE QUERIES


QUERY 1: What is the annual leave policy?



C:\Users\thedo\AppData\Local\Temp\ipykernel_26980\784397839.py:21: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = self.qa_chain({"query": question})


ANSWER:
I don't have information about this in the HR policy document.


SOURCE DOCUMENTS (3 found):
--------------------------------------------------------------------------------

Source 1:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy

Source 2:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy

Source 3:
that goes beyond the traditional aspects of 
collective bargaining in order to share knowledge 
and to jointly find opportunities related to 
important matters such as Creating Shared Value, 
the heal...


QUERY 2: What are the employee benefits?

ANSWER:
The employee benefits at Nestlé include Fixed Pay, Variable Pay, Benefits, Personal Growth and Development, and Work Life Environment. These elements are part of the Total Rewards package that Nestlé focuses on to attract and retain employees. For specific details on the benefits offered, further consultation with the HR department or the company's benefits documentation may be required

## 7. Interactive Chatbot Function

In [10]:
def display_response(response: Dict[str, Any], show_sources: bool = True) -> None:
    """
    Display the chatbot response in a formatted way.
    
    Args:
        response: Response dictionary from the HR Assistant
        show_sources: Whether to display source documents
    """
    if response["error"]:
        print(f"❌ {response['answer']}")
    else:
        print("\n" + "="*80)
        print("HR ASSISTANT RESPONSE")
        print("="*80)
        print(f"\n{response['answer']}")
        
        if show_sources and response["sources"]:
            print("\n" + "-"*80)
            print(f"SOURCE DOCUMENTS ({len(response['sources'])} found):")
            print("-"*80)
            for i, source in enumerate(response["sources"], 1):
                print(f"\nSource {i}:")
                print(source["content"])
                print()


def interactive_chat(num_rounds: int = 3) -> None:
    """
    Run an interactive chat session with the HR Assistant.
    
    Args:
        num_rounds: Number of questions to ask
    """
    if not hr_assistant:
        print("❌ HR Assistant not initialized.")
        return
    
    print("\n" + "="*80)
    print("NESTLÉ HR ASSISTANT - INTERACTIVE MODE")
    print("="*80)
    print("\nAsk questions about Nestlé's HR policies.")
    print("Type 'quit' to exit, 'history' to see conversation history.\n")
    
    for round_num in range(num_rounds):
        user_input = input(f"\nRound {round_num + 1} - Your question: ").strip()
        
        if user_input.lower() == 'quit':
            print("\nThank you for using the HR Assistant. Goodbye!")
            break
        
        if user_input.lower() == 'history':
            history = hr_assistant.get_conversation_history()
            if history:
                print(f"\n{'='*80}")
                print("CONVERSATION HISTORY")
                print(f"{'='*80}")
                for i, item in enumerate(history, 1):
                    print(f"\n{i}. Q: {item['question']}")
                    print(f"   A: {item['answer'][:100]}...")
            else:
                print("\nNo conversation history yet.")
            continue
        
        if not user_input:
            print("Please enter a question.")
            continue
        
        print("\nProcessing your question...")
        response = hr_assistant.answer_question(user_input)
        display_response(response)


# Example usage
print("\n" + "="*80)
print("EXAMPLE: HOW TO USE THE HR ASSISTANT")
print("="*80)
print("""
1. To ask a single question:
   response = hr_assistant.answer_question("What is the leave policy?")
   display_response(response)

2. For interactive chat (uncomment below and run):
   interactive_chat(num_rounds=5)

3. To get conversation history:
   history = hr_assistant.get_conversation_history()
   
4. To clear conversation history:
   hr_assistant.clear_history()
""")

print("\n✓ HR Assistant is ready to answer your questions about Nestlé HR policies!")


EXAMPLE: HOW TO USE THE HR ASSISTANT

1. To ask a single question:
   response = hr_assistant.answer_question("What is the leave policy?")
   display_response(response)

2. For interactive chat (uncomment below and run):
   interactive_chat(num_rounds=5)

3. To get conversation history:
   history = hr_assistant.get_conversation_history()

4. To clear conversation history:
   hr_assistant.clear_history()


✓ HR Assistant is ready to answer your questions about Nestlé HR policies!


## 8. Build Gradio Web Interface

In [11]:
# Gradio is installed in the first cell. Import for the user-friendly chatbot interface (per assignment).
try:
    import gradio as gr
    print("✓ Gradio ready for chatbot interface")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
    import gradio as gr
    print("✓ Gradio installed and imported")

✓ Gradio ready for chatbot interface


In [12]:
# Use Gradio to build user-friendly chatbot interface (per assignment)
import gradio as gr


# Create Gradio chatbot function
def gradio_chatbot_interface(question: str) -> tuple:
    """
    Process a question through the HR Assistant and return formatted response.
    
    Args:
        question: User's question about HR policies
        
    Returns:
        Tuple of (answer, sources_text)
    """
    if not hr_assistant:
        return (
            "❌ Error: HR Assistant not initialized. Please check your OpenAI API key and try again.",
            "No sources available"
        )
    
    if not question.strip():
        return (
            "⚠️ Please enter a question to get started.",
            "No sources available"
        )
    
    # Get response from HR Assistant
    response = hr_assistant.answer_question(question)
    
    # Format answer
    if response["error"]:
        answer_text = f"❌ {response['answer']}"
    else:
        answer_text = f"✓ {response['answer']}"
    
    # Format sources
    if response["sources"]:
        sources_text = "📄 **Source Documents:**\n\n"
        for i, source in enumerate(response["sources"], 1):
            sources_text += f"**Source {i}:**\n{source['content']}\n\n"
    else:
        sources_text = "No source documents found."
    
    return answer_text, sources_text


# Create Gradio Blocks interface with custom layout
if hr_assistant:
    with gr.Blocks(title="Nestlé HR Assistant", theme=gr.themes.Soft()) as demo:
        # Header
        gr.Markdown("""
        # 🏢 Nestlé HR Assistant
        
        **Powered by AI** | Retrieval-Augmented Generation (RAG)
        
        Ask questions about Nestlé's HR policies and get instant, accurate answers based on the official HR policy documents.
        """)
        
        # Main content area
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📝 Ask Your Question")
                user_question = gr.Textbox(
                    label="Your HR Policy Question",
                    placeholder="e.g., What is the annual leave policy? How does maternity leave work?",
                    lines=3,
                    info="Enter any question about Nestlé's HR policies"
                )
                
                # Buttons
                with gr.Row():
                    submit_btn = gr.Button("🔍 Get Answer", variant="primary", size="lg")
                    clear_btn = gr.Button("🔄 Clear", size="lg")
        
        # Response area
        gr.Markdown("### 💡 Response from HR Assistant")
        with gr.Row():
            with gr.Column(scale=2):
                answer_output = gr.Textbox(
                    label="HR Assistant Response",
                    lines=8,
                    interactive=False,
                    info="The AI-generated answer based on HR policy documents"
                )
            
            with gr.Column(scale=1):
                sources_output = gr.Markdown(
                    label="Source Documents",
                    value="No sources available"
                )
        
        # Example questions
        gr.Markdown("### 💬 Example Questions")
        example_questions = [
            "What is the annual leave policy?",
            "What are the employee benefits?",
            "How is sick leave handled?",
            "What is the maternity and paternity policy?",
            "What are the working hours?",
            "How are performance appraisals conducted?",
            "What is the code of conduct?",
        ]
        
        gr.Examples(
            examples=[[q] for q in example_questions],
            inputs=user_question,
            label="Click an example to try it"
        )
        
        # Info section
        gr.Markdown("""
        ### ℹ️ How It Works
        
        1. **Ask a Question**: Type your HR policy question in the text field
        2. **Get AI Response**: The chatbot retrieves relevant documents and generates an answer
        3. **View Sources**: See which HR policy documents were used to answer your question
        4. **Verify Information**: Cross-check answers with the provided source documents
        
        This chatbot uses **Retrieval-Augmented Generation (RAG)** technology:
        - 🔍 **Retrieval**: Searches the HR policy document for relevant information
        - 🧠 **Generation**: Uses GPT-3.5 Turbo to create natural, accurate responses
        - 📚 **Sources**: Shows which parts of the document were used
        """)
        
        # Event handlers
        submit_btn.click(
            fn=gradio_chatbot_interface,
            inputs=user_question,
            outputs=[answer_output, sources_output],
            api_name="get_hr_answer"
        )
        
        clear_btn.click(
            fn=lambda: ("", "No sources available"),
            outputs=[answer_output, sources_output]
        )
        
        # Enter key support
        user_question.submit(
            fn=gradio_chatbot_interface,
            inputs=user_question,
            outputs=[answer_output, sources_output]
        )
    
    print("✓ Gradio interface created successfully!")
    print("\n" + "="*80)
    print("LAUNCHING WEB INTERFACE")
    print("="*80)
    print("\nThe HR Assistant web interface will open in your browser.")
    print("If it doesn't open automatically, visit: http://localhost:7860")
    print("\nFeatures:")
    print("  - Ask HR policy questions in natural language")
    print("  - Get AI-powered responses with source documents")
    print("  - Try example questions with one click")
    print("  - Copy responses easily")
    print("\nTo stop the server, press Ctrl+C in the terminal.")
    print("="*80 + "\n")
    
    # Launch the interface
    demo.launch(share=False, show_error=True)

else:
    print("❌ Cannot create Gradio interface without HR Assistant.")
    print("Please ensure all previous setup steps completed successfully.")

C:\Users\thedo\AppData\Local\Temp\ipykernel_26980\1950868413.py:50: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Nestlé HR Assistant", theme=gr.themes.Soft()) as demo:


✓ Gradio interface created successfully!

LAUNCHING WEB INTERFACE

The HR Assistant web interface will open in your browser.
If it doesn't open automatically, visit: http://localhost:7860

Features:
  - Ask HR policy questions in natural language
  - Get AI-powered responses with source documents
  - Try example questions with one click
  - Copy responses easily

To stop the server, press Ctrl+C in the terminal.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
